# Week 5: Community Detection — Assignment

**Learning objectives** — In this assignment you will:

- Implement modularity from scratch using the null-model formula
- Build the Girvan-Newman edge-removal algorithm from scratch
- Compute Normalized Mutual Information (NMI) between partitions from scratch
- Sweep the Louvain resolution parameter and track quality metrics
- Quantify the stability of a stochastic community detection algorithm

## Grading

| Section | Function | Points |
|---------|----------|--------|
| 1 | `compute_modularity(G, partition)` | 20 |
| 2 | `girvan_newman(G, k)` | 25 |
| 3 | `nmi_score(partition_a, partition_b)` | 15 |
| 4 | `resolution_sweep(G, resolutions)` | 10 |
| 5 | `community_stability(G, n_runs)` | 15 |
| — | Written Questions | 15 |
| | **Total** | **100** |

## Before You Start

This assignment builds on the Week 5 lab. Make sure you are comfortable with:

- **Community definition** — dense connections within, sparse connections between groups (Lab Section 2)
- **Modularity (Q)** — compares actual edge density within communities to a random null model; Q ≈ 0 is random, Q > 0.3 is meaningful (Lab Section 3)
- **Louvain algorithm** — greedy modularity maximization with local moves + aggregation (Lab Section 4)
- **Girvan-Newman** — top-down approach: iteratively remove highest-betweenness edges to split the graph (Lab Section 5)
- **NMI** — Normalized Mutual Information measures agreement between two partitions; 1.0 = perfect match, 0.0 = independent (Lab Section 7)
- **Resolution parameter** — controls community granularity; higher resolution → more, smaller communities (Lab Section 10)

### Implementation constraints

The following functions are **banned** from your solutions:

| Banned function | Sections |
|----------------|----------|
| `nx.community.modularity` | 1, 4 |
| `nx.community.girvan_newman` | 2 |
| `sklearn.metrics.normalized_mutual_info_score`, `sklearn.metrics.mutual_info_score` | 3, 5 |

You **may** use: `G.neighbors()`, `G.nodes()`, `G.edges()`, `G.degree()`, `G.number_of_edges()`, `G.number_of_nodes()`, `nx.Graph()`, `nx.edge_betweenness_centrality()`, `nx.connected_components()`, `nx.community.louvain_communities()`, `nx.average_clustering()`, `np.log()`, `collections.Counter`.

**Important**: Later sections build on earlier ones. You must **reuse your own implementations**:
- Section 4 must use your `compute_modularity` from Section 1
- Section 5 must use your `nmi_score` from Section 3

In [1]:
import networkx as nx
import numpy as np
from netsci.loaders import load_graph
from netsci.utils import SEED

In [2]:
G_karate = load_graph("karate")
G_football = load_graph("football")

# Ground truth for karate (2 factions)
gt_karate = [
    {n for n in G_karate.nodes() if G_karate.nodes[n].get("club") == "Mr. Hi"},
    {n for n in G_karate.nodes() if G_karate.nodes[n].get("club") == "Officer"},
]

# Ground truth for football (12 conferences)
_conf_map = {}
for n in G_football.nodes():
    c = G_football.nodes[n]["conference"]
    _conf_map.setdefault(c, set()).add(n)
gt_football = list(_conf_map.values())

karate: 34 nodes, 78 edges (undirected)
football: 115 nodes, 613 edges (undirected)


---
## Section 1: Modularity from Scratch (20 pts)

Implement the **unweighted** modularity formula:

$$Q = \frac{1}{2m} \sum_{ij} \left[ A_{ij} - \frac{k_i k_j}{2m} \right] \delta(c_i, c_j)$$

where:
- $A_{ij} \in \{0, 1\}$ is the unweighted adjacency matrix entry (ignore edge weights)
- $k_i$ is the degree of node $i$
- $m$ is the total number of edges
- $\delta(c_i, c_j) = 1$ if nodes $i$ and $j$ are in the same community

**Equivalent per-community form** (easier to implement):

$$Q = \sum_{c} \left[ \frac{L_c}{m} - \left( \frac{d_c}{2m} \right)^2 \right]$$

where $L_c$ is the number of internal edges in community $c$ and $d_c = \sum_{i \in c} k_i$.

**Do not use** `nx.community.modularity`.

In [3]:
def compute_modularity(G, partition):
    """Compute unweighted modularity of a partition from scratch.

    Parameters
    ----------
    G : nx.Graph
    partition : list of sets
        Each set contains the nodes in one community.

    Returns
    -------
    float
    """
    # YOUR CODE HERE
    m = G.number_of_edges()
    Q = 0.0
    for community in partition:
        for u in community:
            for v in community:
                A_uv = 1 if G.has_edge(u, v) else 0
                k_u = G.degree(u)
                k_v = G.degree(v)
                Q += A_uv - (k_u * k_v) / (2 * m)
    Q /= (2 * m)
    return Q

In [4]:
# --- Validation ---
# Note: we use weight=None because the formula above is unweighted
_Q = compute_modularity(G_karate, gt_karate)
_Q_nx = nx.community.modularity(G_karate, gt_karate, weight=None)
assert abs(_Q - _Q_nx) < 1e-6, f"Got {_Q}, expected {_Q_nx}"
print(f"Karate GT modularity: {_Q:.6f} (expected: {_Q_nx:.6f})")

# Test with Louvain output on football
_louv = list(nx.community.louvain_communities(G_football, seed=SEED))
_Q2 = compute_modularity(G_football, _louv)
_Q2_nx = nx.community.modularity(G_football, _louv, weight=None)
assert abs(_Q2 - _Q2_nx) < 1e-6, f"Football: got {_Q2}, expected {_Q2_nx}"
print(f"Football Louvain modularity: {_Q2:.6f}")

# Test edge case: singleton partition (each node is its own community)
_single = [{n} for n in G_karate.nodes()]
_Q3 = compute_modularity(G_karate, _single)
_Q3_nx = nx.community.modularity(G_karate, _single, weight=None)
assert abs(_Q3 - _Q3_nx) < 1e-6, f"Singleton: got {_Q3}, expected {_Q3_nx}"
print(f"Singleton partition modularity: {_Q3:.6f} (should be negative)")

# Test: one giant community (all nodes together)
_all_one = [set(G_karate.nodes())]
_Q4 = compute_modularity(G_karate, _all_one)
assert abs(_Q4 - 0.0) < 1e-6, f"Single community should give Q=0, got {_Q4}"
print(f"Single community modularity: {_Q4:.6f} (should be 0)")
print("Section 1 passed!")

Karate GT modularity: 0.358235 (expected: 0.358235)
Football Louvain modularity: 0.604570
Singleton partition modularity: -0.049803 (should be negative)
Single community modularity: -0.000000 (should be 0)
Section 1 passed!


---
## Section 2: Girvan-Newman from Scratch (25 pts)

Implement the Girvan-Newman algorithm **from scratch**. **Do NOT** call `nx.community.girvan_newman`.

The algorithm:

1. **Start** with a copy of the input graph
2. **Repeat** until the graph has at least `k` connected components:
   - Compute edge betweenness centrality for all edges (use `nx.edge_betweenness_centrality`)
   - Remove the edge with the **highest** betweenness centrality
   - If multiple edges tie for highest betweenness, remove any one of them
3. **Return** the connected components as a list of sets

**Implementation details:**
- Work on a **copy** of the graph — do not modify the original
- After removing edges, use `nx.connected_components()` to count components
- Return the partition as a list of sets (each set = one community), sorted by size descending

In [5]:
def girvan_newman(G, k=2):
    """Split a graph into k communities using Girvan-Newman.

    Do NOT call nx.community.girvan_newman.

    Parameters
    ----------
    G : nx.Graph
        Must be connected.
    k : int
        Target number of communities.

    Returns
    -------
    list of sets — partition into k communities, sorted by size descending
    """
    # YOUR CODE HERE
    if k <= 0:
        raise ValueError("k must be a positive integer")
    if k == 1:
        return [set(G.nodes())]
    if not nx.is_connected(G):
        raise ValueError("Input graph must be connected")
    G_copy = G.copy()
    while len(list(nx.connected_components(G_copy))) < k:
        edge_betweenness = nx.edge_betweenness_centrality(G_copy)
        max_betweenness = max(edge_betweenness.values())
        edges_to_remove = [edge for edge, betweenness in edge_betweenness.items() if betweenness == max_betweenness]
        G_copy.remove_edges_from(edges_to_remove)
    communities = list(nx.connected_components(G_copy))
    communities.sort(key=len, reverse=True)
    return communities

In [6]:
# --- Validation ---
# Karate club: split into 2
_gn2 = girvan_newman(G_karate, k=2)
assert isinstance(_gn2, list)
assert len(_gn2) == 2, f"Expected 2 communities, got {len(_gn2)}"
assert all(isinstance(c, set) for c in _gn2)
_all_nodes_gn = set()
for c in _gn2:
    _all_nodes_gn |= c
assert _all_nodes_gn == set(G_karate.nodes()), "All nodes must be assigned"

# Should produce a decent partition
_Q_gn = nx.community.modularity(G_karate, _gn2, weight=None)
assert _Q_gn > 0.3, f"GN modularity {_Q_gn:.3f} too low for karate (expected > 0.3)"
print(f"GN(k=2) on karate: sizes={sorted([len(c) for c in _gn2], reverse=True)}, Q={_Q_gn:.4f}")

# Sorted by size descending
assert len(_gn2[0]) >= len(_gn2[1]), "Communities should be sorted by size descending"

# Original graph unchanged
assert G_karate.number_of_edges() == 78, "Original graph must not be modified"

# Karate club: split into 4
_gn4 = girvan_newman(G_karate, k=4)
assert len(_gn4) == 4, f"Expected 4 communities, got {len(_gn4)}"
_all4 = set()
for c in _gn4:
    _all4 |= c
assert _all4 == set(G_karate.nodes())
_Q_gn4 = nx.community.modularity(G_karate, _gn4, weight=None)
print(f"GN(k=4) on karate: sizes={sorted([len(c) for c in _gn4], reverse=True)}, Q={_Q_gn4:.4f}")

# Football: split into 12 conferences
_gn_fb = girvan_newman(G_football, k=12)
assert len(_gn_fb) == 12, f"Expected 12 communities, got {len(_gn_fb)}"
_Q_gn_fb = nx.community.modularity(G_football, _gn_fb, weight=None)
assert _Q_gn_fb > 0.4, f"GN modularity on football {_Q_gn_fb:.3f} too low"
print(f"GN(k=12) on football: Q={_Q_gn_fb:.4f}")
print("Section 2 passed!")

GN(k=2) on karate: sizes=[19, 15], Q=0.3600
GN(k=4) on karate: sizes=[18, 10, 5, 1], Q=0.3632
GN(k=12) on football: Q=0.5973
Section 2 passed!


---
## Section 3: Normalized Mutual Information (15 pts)

Implement NMI from scratch. **Do NOT** use `sklearn.metrics.normalized_mutual_info_score` or any library NMI function.

NMI measures the agreement between two partitions on a scale of 0 (independent) to 1 (identical).

$$\text{NMI}(U, V) = \frac{2 \cdot I(U; V)}{H(U) + H(V)}$$

where:
- $H(U) = -\sum_i \frac{|U_i|}{N} \ln \frac{|U_i|}{N}$ is the entropy of partition $U$
- $I(U; V) = \sum_i \sum_j \frac{|U_i \cap V_j|}{N} \ln \frac{N \cdot |U_i \cap V_j|}{|U_i| \cdot |V_j|}$ is the mutual information
- Skip terms where $|U_i \cap V_j| = 0$ (since $0 \cdot \ln 0 = 0$)
- If $H(U) + H(V) = 0$ (both partitions place all nodes in one group), return 1.0

In [7]:
def nmi_score(partition_a, partition_b):
    """Compute Normalized Mutual Information between two partitions.

    Do NOT use sklearn.metrics.normalized_mutual_info_score or any
    library NMI function.

    Parameters
    ----------
    partition_a : list of sets
    partition_b : list of sets

    Returns
    -------
    float (between 0 and 1)
    """
    # Build node -> community index maps.
    node_to_comm_a = {}
    for i, community in enumerate(partition_a):
        for node in community:
            node_to_comm_a[node] = i

    node_to_comm_b = {}
    for j, community in enumerate(partition_b):
        for node in community:
            node_to_comm_b[node] = j

    all_nodes = set(node_to_comm_a) | set(node_to_comm_b)
    N = len(all_nodes)
    if N == 0:
        return 1.0

    contingency = np.zeros((len(partition_a), len(partition_b)), dtype=float)
    for node in all_nodes:
        i = node_to_comm_a.get(node)
        j = node_to_comm_b.get(node)
        if i is not None and j is not None:
            contingency[i, j] += 1.0

    # Convert to probabilities.
    p_ij = contingency / N
    p_i = p_ij.sum(axis=1)
    p_j = p_ij.sum(axis=0)

    # Mutual information: sum only over non-zero cells.
    mi = 0.0
    for i in range(p_ij.shape[0]):
        for j in range(p_ij.shape[1]):
            if p_ij[i, j] > 0:
                mi += p_ij[i, j] * np.log(p_ij[i, j] / (p_i[i] * p_j[j]))

    # Entropies: sum only over non-zero probabilities.
    h_a = -np.sum([p * np.log(p) for p in p_i if p > 0])
    h_b = -np.sum([p * np.log(p) for p in p_j if p > 0])

    if h_a + h_b == 0:
        return 1.0

    return float(2.0 * mi / (h_a + h_b))

In [8]:
# --- Validation ---
from sklearn.metrics import normalized_mutual_info_score as _sklearn_nmi

def _to_labels(partition, nodes):
    m = {}
    for i, c in enumerate(partition):
        for n in c:
            m[n] = i
    return [m[n] for n in nodes]

# Perfect match should be 1.0
_nmi_perfect = nmi_score(gt_karate, gt_karate)
assert abs(_nmi_perfect - 1.0) < 1e-6, f"Perfect NMI should be 1.0, got {_nmi_perfect}"
print(f"NMI(GT, GT) = {_nmi_perfect:.4f}")

# All-in-one vs all-in-one should be 1.0 (edge case)
_all_one_a = [set(G_karate.nodes())]
_all_one_b = [set(G_karate.nodes())]
_nmi_trivial = nmi_score(_all_one_a, _all_one_b)
assert abs(_nmi_trivial - 1.0) < 1e-6, f"Trivial NMI should be 1.0, got {_nmi_trivial}"

# Louvain vs ground truth on karate
_louv_k = list(nx.community.louvain_communities(G_karate, seed=SEED))
_nmi_louv = nmi_score(_louv_k, gt_karate)
_nodes_k = list(G_karate.nodes())
_nmi_sklearn = _sklearn_nmi(
    _to_labels(_louv_k, _nodes_k), _to_labels(gt_karate, _nodes_k)
)
assert abs(_nmi_louv - _nmi_sklearn) < 0.05, (
    f"NMI(Louvain, GT) = {_nmi_louv:.4f}, sklearn says {_nmi_sklearn:.4f}"
)
print(f"NMI(Louvain, GT) on karate = {_nmi_louv:.4f} (sklearn: {_nmi_sklearn:.4f})")

# Football: Louvain vs conference ground truth
_louv_fb = list(nx.community.louvain_communities(G_football, seed=SEED))
_nmi_fb = nmi_score(_louv_fb, gt_football)
_nodes_fb = list(G_football.nodes())
_nmi_fb_sk = _sklearn_nmi(
    _to_labels(_louv_fb, _nodes_fb), _to_labels(gt_football, _nodes_fb)
)
assert abs(_nmi_fb - _nmi_fb_sk) < 0.05, (
    f"Football NMI = {_nmi_fb:.4f}, sklearn = {_nmi_fb_sk:.4f}"
)
print(f"NMI(Louvain, GT) on football = {_nmi_fb:.4f}")

# NMI should be > 0 but < 1 for imperfect match
assert 0 < _nmi_louv < 1.0, f"Expected 0 < NMI < 1, got {_nmi_louv}"
print("Section 3 passed!")

NMI(GT, GT) = 1.0000
NMI(Louvain, GT) on karate = 0.5942 (sklearn: 0.5942)
NMI(Louvain, GT) on football = 0.8903
Section 3 passed!


---
## Section 4: Resolution Sweep (10 pts)

Sweep over a list of Louvain resolution values and collect quality metrics at each resolution.

**Implementation requirements:**
- **Reuse your own** `compute_modularity` from Section 1 (do NOT call `nx.community.modularity`)
- Use `nx.community.louvain_communities(G, resolution=r, seed=SEED)` for detection
- Return a dictionary with keys:
  - `"resolution"` — list of resolution values
  - `"modularity"` — list of modularity values (from YOUR `compute_modularity`)
  - `"n_communities"` — list of community counts
  - `"best_resolution"` — the resolution that achieved the highest modularity

In [9]:
def resolution_sweep(G, resolutions):
    """Sweep Louvain resolution and collect quality metrics.

    Reuse your compute_modularity() from Section 1.
    Do NOT call nx.community.modularity.

    Parameters
    ----------
    G : nx.Graph
    resolutions : list of float

    Returns
    -------
    dict with keys 'resolution', 'modularity', 'n_communities', 'best_resolution'
    """
    # YOUR CODE HERE
    results = {
        'resolution': [],
        'modularity': [],
        'n_communities': []
    }
    best_modularity = -1.0
    best_resolution = None
    for res in resolutions:
        partition = list(nx.community.louvain_communities(G, resolution=res, seed=SEED))
        mod = compute_modularity(G, partition)
        results['resolution'].append(res)
        results['modularity'].append(mod)
        results['n_communities'].append(len(partition))
        if mod > best_modularity:
            best_modularity = mod
            best_resolution = res
    results['best_resolution'] = best_resolution
    return results

In [ ]:

d


# --- Validation ---
_res = [0.5, 0.8, 1.0, 1.2, 1.5, 2.0, 3.0]
_result = resolution_sweep(G_football, _res)

assert set(_result.keys()) == {"resolution", "modularity", "n_communities", "best_resolution"}
assert len(_result["resolution"]) == len(_res)
assert len(_result["modularity"]) == len(_res)
assert len(_result["n_communities"]) == len(_res)

# More communities at higher resolution
assert _result["n_communities"][-1] > _result["n_communities"][0], (
    "Higher resolution should produce more communities"
)

# Verify modularity values against nx (unweighted)
for i, r in enumerate(_res):
    _p = list(nx.community.louvain_communities(G_football, resolution=r, seed=SEED))
    _q_expected = nx.community.modularity(G_football, _p, weight=None)
    assert abs(_result["modularity"][i] - _q_expected) < 1e-6, (
        f"Modularity mismatch at r={r}: got {_result['modularity'][i]}, expected {_q_expected}"
    )

# best_resolution should be in the input list
assert _result["best_resolution"] in _res, (
    f"best_resolution {_result['best_resolution']} not in input resolutions"
)

# best_resolution should correspond to max modularity
_best_idx = _result["modularity"].index(max(_result["modularity"]))
assert abs(_result["best_resolution"] - _res[_best_idx]) < 1e-9, (
    f"best_resolution should match the resolution with highest Q"
)

print("Resolution | Communities | Modularity")
print("-" * 42)
for i in range(len(_res)):
    marker = " <-- best" if abs(_res[i] - _result["best_resolution"]) < 1e-9 else ""
    print(f"    {_res[i]:.1f}    |     {_result['n_communities'][i]:2d}      | {_result['modularity'][i]:.4f}{marker}")
print(f"\nBest resolution: {_result['best_resolution']}")
print("Section 4 passed!")

Resolution | Communities | Modularity
------------------------------------------
    0.5    |      6      | 0.5828
    0.8    |      7      | 0.6006
    1.0    |     10      | 0.6046 <-- best
    1.2    |     10      | 0.6044
    1.5    |     12      | 0.6005
    2.0    |     12      | 0.6005
    3.0    |     12      | 0.6005

Best resolution: 1.0
Section 4 passed!


---
## Section 5: Community Stability (15 pts)

Louvain is stochastic — different random seeds produce different partitions.
Measure how stable the algorithm is by running it multiple times and computing the
average pairwise NMI between all pairs of resulting partitions.

**Implementation requirements:**
- Run `nx.community.louvain_communities(G, seed=i)` for `i = 0, 1, ..., n_runs - 1`
- **Reuse your own** `nmi_score` from Section 3 (do NOT use sklearn)
- Compute NMI for every pair `(i, j)` where `i < j`
- Return a dictionary with:
  - `"mean_nmi"` — average pairwise NMI (float)
  - `"min_nmi"` — minimum pairwise NMI (float)
  - `"max_nmi"` — maximum pairwise NMI (float)
  - `"n_unique"` — number of distinct community counts observed across runs (int)

In [13]:
def community_stability(G, n_runs=10):
    """Measure stability of Louvain across multiple runs.

    Reuse your nmi_score() from Section 3.
    Do NOT use sklearn.metrics.normalized_mutual_info_score.

    Parameters
    ----------
    G : nx.Graph
    n_runs : int

    Returns
    -------
    dict with keys 'mean_nmi', 'min_nmi', 'max_nmi', 'n_unique'
    """
    if n_runs < 2:
        raise ValueError("n_runs must be at least 2")

    partitions = []
    for i in range(n_runs):
        partition = list(nx.community.louvain_communities(G, seed=i))
        partitions.append(partition)

    nmi_values = []
    for i in range(n_runs):
        for j in range(i + 1, n_runs):
            nmi = nmi_score(partitions[i], partitions[j])
            nmi_values.append(nmi)

    mean_nmi = float(np.mean(nmi_values))
    min_nmi = float(np.min(nmi_values))
    max_nmi = float(np.max(nmi_values))

    # Number of distinct community counts observed across runs.
    n_unique = len({len(p) for p in partitions})

    return {
        'mean_nmi': mean_nmi,
        'min_nmi': min_nmi,
        'max_nmi': max_nmi,
        'n_unique': n_unique
    }

In [14]:
# --- Validation ---
_stab_k = community_stability(G_karate, n_runs=10)
assert set(_stab_k.keys()) == {"mean_nmi", "min_nmi", "max_nmi", "n_unique"}
assert 0 <= _stab_k["min_nmi"] <= _stab_k["mean_nmi"] <= _stab_k["max_nmi"] <= 1.0
assert isinstance(_stab_k["n_unique"], int) and _stab_k["n_unique"] >= 1

print(f"Karate stability (10 runs):")
print(f"  Mean NMI: {_stab_k['mean_nmi']:.4f}")
print(f"  Min NMI:  {_stab_k['min_nmi']:.4f}")
print(f"  Max NMI:  {_stab_k['max_nmi']:.4f}")
print(f"  Unique community counts: {_stab_k['n_unique']}")

_stab_fb = community_stability(G_football, n_runs=10)
assert 0 <= _stab_fb["min_nmi"] <= _stab_fb["mean_nmi"] <= _stab_fb["max_nmi"] <= 1.0

# Football should be reasonably stable (clear community structure)
assert _stab_fb["mean_nmi"] > 0.85, (
    f"Football mean NMI {_stab_fb['mean_nmi']:.3f} too low — expected > 0.85"
)
print(f"\nFootball stability (10 runs):")
print(f"  Mean NMI: {_stab_fb['mean_nmi']:.4f}")
print(f"  Min NMI:  {_stab_fb['min_nmi']:.4f}")
print(f"  Max NMI:  {_stab_fb['max_nmi']:.4f}")
print(f"  Unique community counts: {_stab_fb['n_unique']}")

# Verify against sklearn
from sklearn.metrics import normalized_mutual_info_score as _sk_nmi

def _to_labels(partition, nodes):
    m = {}
    for i, c in enumerate(partition):
        for n in c:
            m[n] = i
    return [m[n] for n in nodes]

_parts = [list(nx.community.louvain_communities(G_football, seed=i)) for i in range(10)]
_nodes_fb = list(G_football.nodes())
_sk_nmis = []
for i in range(10):
    for j in range(i + 1, 10):
        _sk_nmis.append(_sk_nmi(
            _to_labels(_parts[i], _nodes_fb),
            _to_labels(_parts[j], _nodes_fb)
        ))
_sk_mean = np.mean(_sk_nmis)
assert abs(_stab_fb["mean_nmi"] - _sk_mean) < 0.05, (
    f"Mean NMI {_stab_fb['mean_nmi']:.4f} differs from sklearn {_sk_mean:.4f}"
)
print(f"  (sklearn verification: mean={_sk_mean:.4f})")
print("Section 5 passed!")

Karate stability (10 runs):
  Mean NMI: 0.9214
  Min NMI:  0.6880
  Max NMI:  1.0000
  Unique community counts: 2

Football stability (10 runs):
  Mean NMI: 0.9629
  Min NMI:  0.9198
  Max NMI:  1.0000
  Unique community counts: 2
  (sklearn verification: mean=0.9629)
Section 5 passed!


---
## Written Questions (15 pts)

### Question 1 (5 pts)

Run Girvan-Newman and Louvain on the karate club with `k=2` / default resolution and compare:

(a) Report the community sizes and modularity Q for each algorithm. Are the partitions identical? If not, which nodes are assigned differently?

(b) Girvan-Newman is $O(m^2 n)$ while Louvain is nearly $O(n \log n)$. Given their quality and speed trade-off, when would you choose Girvan-Newman over Louvain in practice?

(c) Compute the NMI between the two partitions (using your `nmi_score`). Is the disagreement large or small? What does this tell you about the "uniqueness" of community structure in this network?

**You must include numerical values from your code.** Show the code cell you used to compute them.

*Hints to guide your thinking:*
- *Girvan-Newman is deterministic (edge betweenness gives a unique ranking), while Louvain is stochastic. What are the implications?*
- *Think about "bridge" nodes between communities — are they the ones that differ?*
- *Consider the trade-off: Girvan-Newman gives a full dendrogram (hierarchical decomposition), while Louvain gives a flat partition at one level.*

**Your Answer:**

Code used: see **Cell 25** directly below.

(a) **Community sizes and modularity**

- Girvan-Newman (`k=2`): sizes = `[19, 15]`, modularity $Q = 0.359961$.
- Louvain (default resolution): sizes = `[14, 10, 6, 4]`, modularity $Q = 0.390450$.

The partitions are **not identical**.
They differ in granularity (2 communities vs 4 communities), and a direct label-aligned node comparison reports differing nodes:
`[1, 2, 3, 7, 12, 13, 24, 25, 28, 31]`.

(b) **When to choose Girvan-Newman vs Louvain**

I would choose **Girvan-Newman** when I need interpretability and hierarchy (a dendrogram-like split process), or when the network is small enough that runtime is not a bottleneck and I care about edge-bridge structure explicitly.
I would choose **Louvain** for larger graphs or repeated experiments because it is much faster and here also gives higher modularity.

(c) **NMI and interpretation**

The NMI between Girvan-Newman and Louvain is:

$$
\mathrm{NMI}(\text{GN}, \text{Louvain}) = 0.616132
$$

This is a **moderate** agreement (not near 1), so disagreement is meaningful but not extreme.
Interpretation: the karate network has a recognizable community signal, but the exact partition is not unique; boundary/bridge nodes can be assigned differently depending on algorithmic bias and resolution scale.

In [15]:
# Question 1 support code: GN vs Louvain on Karate (k=2 / default resolution)

# Compute partitions
q1_gn = girvan_newman(G_karate, k=2)
q1_louv = list(nx.community.louvain_communities(G_karate, seed=SEED))

# Compute sizes and modularity (using our own function)
q1_gn_sizes = sorted([len(c) for c in q1_gn], reverse=True)
q1_louv_sizes = sorted([len(c) for c in q1_louv], reverse=True)
q1_q_gn = compute_modularity(G_karate, q1_gn)
q1_q_louv = compute_modularity(G_karate, q1_louv)

# Canonicalize labels so partitions can be compared node-by-node
q1_nodes_sorted = sorted(G_karate.nodes())
q1_gn_sorted_comms = sorted([sorted(c) for c in q1_gn], key=lambda c: (-len(c), c))
q1_louv_sorted_comms = sorted([sorted(c) for c in q1_louv], key=lambda c: (-len(c), c))

q1_gn_label = {}
for idx, comm in enumerate(q1_gn_sorted_comms):
    for node in comm:
        q1_gn_label[node] = idx

q1_louv_label = {}
for idx, comm in enumerate(q1_louv_sorted_comms):
    for node in comm:
        q1_louv_label[node] = idx

q1_diff_nodes = [n for n in q1_nodes_sorted if q1_gn_label[n] != q1_louv_label[n]]

# NMI between partitions
q1_nmi = nmi_score(q1_gn, q1_louv)

print("Girvan-Newman sizes:", q1_gn_sizes)
print("Louvain sizes:", q1_louv_sizes)
print(f"Q (Girvan-Newman): {q1_q_gn:.6f}")
print(f"Q (Louvain):        {q1_q_louv:.6f}")
print("Partitions identical:", len(q1_diff_nodes) == 0)
print("Differing nodes:", q1_diff_nodes)
print(f"NMI(GN, Louvain): {q1_nmi:.6f}")

Girvan-Newman sizes: [19, 15]
Louvain sizes: [14, 10, 6, 4]
Q (Girvan-Newman): 0.359961
Q (Louvain):        0.390450
Partitions identical: False
Differing nodes: [1, 2, 3, 7, 12, 13, 24, 25, 28, 31]
NMI(GN, Louvain): 0.616132


### Question 2 (5 pts)

Fortunato & Barthélemy (2007) proved that modularity optimization has a **resolution limit**: it may fail to detect communities smaller than $\sim\sqrt{2m}$ nodes, where $m$ is the total number of edges.

(a) For the football network, compute $\sqrt{2m}$. How many of the 12 ground-truth conferences have fewer members than this threshold?

(b) Yet Louvain still finds ~10 communities with Q > 0.55 on this network. Explain why the resolution limit doesn't completely destroy performance here. (*Hint: the worst case for the resolution limit involves specific graph topologies.*)

(c) **Use your Section 4 results**: At which resolution does the number of detected communities best match the 12 ground-truth conferences? Is this the same resolution that maximizes modularity? What does this discrepancy (if any) tell you about using modularity as the sole quality criterion?

*Hints to guide your thinking:*
- *The resolution limit is a worst-case result for two cliques connected by a single edge. Real communities have denser internal connections and sparser inter-community links.*
- *Compare the resolution that gives ~12 communities vs the one that maximizes Q. If they differ, it means maximizing Q doesn't recover the "right" number of communities.*
- *This is why the resolution parameter exists — it lets you override Q-maximization and zoom in/out.*

**Your Answer:**

Code used: see **Cell 29** directly below.

(a) For football, the number of edges is $m=613$, so:

$$
\sqrt{2m}=\sqrt{1226}=35.0143
$$

Ground-truth conference sizes are:
`[5, 7, 8, 8, 9, 10, 10, 10, 11, 12, 12, 13]`.
All **12/12** conferences are smaller than $\sqrt{2m}$.

(b) Even though the resolution-limit theorem warns that small modules can be merged, it is a worst-case phenomenon tied to specific topologies (for example, weakly connected clique chains). The football graph has comparatively strong within-conference density and sparse between-conference links, so modularity-based methods still recover meaningful mesoscale structure and high-quality partitions (here $Q \approx 0.60$).

(c) From the Section 4-style sweep:
- Closest match to 12 communities occurs at $r=1.5$ (also at $2.0$ and $3.0$), with $n_{\text{communities}}=12$ and $Q=0.6005$.
- Maximum modularity occurs at $r=1.0$, with $n_{\text{communities}}=10$ and $Q=0.6046$.

So the resolution that best matches ground truth is **not** the one that maximizes modularity. This shows modularity alone is not a sufficient objective when the target is semantic/ground-truth recovery; you should tune resolution and evaluate with external criteria (for example, known labels, NMI, or task utility).

In [16]:
# Question 2 support code: resolution limit + resolution sweep interpretation

# (a) Resolution-limit threshold for football
q2_m = G_football.number_of_edges()
q2_threshold = np.sqrt(2 * q2_m)
q2_conf_sizes = sorted([len(c) for c in gt_football])
q2_n_small = sum(1 for s in q2_conf_sizes if s < q2_threshold)

# (c) Use Section 4-style sweep results
q2_resolutions = [0.5, 0.8, 1.0, 1.2, 1.5, 2.0, 3.0]
q2_sweep = resolution_sweep(G_football, q2_resolutions)
q2_n_comms = q2_sweep["n_communities"]
q2_mods = q2_sweep["modularity"]

# Resolution whose community count is closest to 12.
q2_target = 12
q2_dist = [abs(nc - q2_target) for nc in q2_n_comms]
q2_min_dist = min(q2_dist)
q2_best_match_idxs = [i for i, d in enumerate(q2_dist) if d == q2_min_dist]

# Tie-break by highest modularity among equally close matches.
q2_best_match_idx = max(q2_best_match_idxs, key=lambda i: q2_mods[i])
q2_best_match_res = q2_resolutions[q2_best_match_idx]
q2_best_match_nc = q2_n_comms[q2_best_match_idx]
q2_best_match_q = q2_mods[q2_best_match_idx]

# Resolution that maximizes modularity.
q2_qmax_idx = int(np.argmax(q2_mods))
q2_qmax_res = q2_resolutions[q2_qmax_idx]
q2_qmax_nc = q2_n_comms[q2_qmax_idx]
q2_qmax = q2_mods[q2_qmax_idx]

print(f"Football edges m = {q2_m}")
print(f"sqrt(2m) = {q2_threshold:.4f}")
print(f"Conference sizes (sorted): {q2_conf_sizes}")
print(f"# conferences smaller than sqrt(2m): {q2_n_small} out of {len(gt_football)}")
print()
print("Resolution sweep summary:")
for r, nc, q in zip(q2_resolutions, q2_n_comms, q2_mods):
    print(f"  r={r:>3}: n_communities={nc:>2}, Q={q:.4f}")
print()
print("Closest to 12 communities:")
print(f"  resolution={q2_best_match_res}, n_communities={q2_best_match_nc}, Q={q2_best_match_q:.4f}")
print("Max modularity:")
print(f"  resolution={q2_qmax_res}, n_communities={q2_qmax_nc}, Q={q2_qmax:.4f}")

Football edges m = 613
sqrt(2m) = 35.0143
Conference sizes (sorted): [5, 7, 8, 8, 9, 10, 10, 10, 11, 12, 12, 13]
# conferences smaller than sqrt(2m): 12 out of 12

Resolution sweep summary:
  r=0.5: n_communities= 6, Q=0.5828
  r=0.8: n_communities= 7, Q=0.6006
  r=1.0: n_communities=10, Q=0.6046
  r=1.2: n_communities=10, Q=0.6044
  r=1.5: n_communities=12, Q=0.6005
  r=2.0: n_communities=12, Q=0.6005
  r=3.0: n_communities=12, Q=0.6005

Closest to 12 communities:
  resolution=3.0, n_communities=12, Q=0.6005
Max modularity:
  resolution=1.0, n_communities=10, Q=0.6046


### Question 3 (5 pts)

Your Section 5 measures the stability of Louvain across runs.

(a) Report the mean, min, and max pairwise NMI for both karate and football. Which network has more stable community assignments? Why? (*Think about the structure of each network.*)

(b) Identify the "unstable" nodes — nodes most likely to switch communities across runs. What structural property do these nodes share? (*Hint: think about their position relative to community boundaries.*)

(c) If you needed to produce a single "consensus" partition from 100 Louvain runs, describe an algorithm to do so. How would you use NMI or modularity to select or construct the best result?

*Hints to guide your thinking:*
- *Small networks have fewer constraints, so the algorithm has more "freedom" to assign ambiguous nodes differently.*
- *Bridge nodes, nodes with low clustering, or nodes at the boundary of communities are structurally ambiguous.*
- *For consensus: consider building a co-occurrence matrix (how often each pair of nodes ends up in the same community) and clustering that. Alternatively, pick the run whose partition has the highest average NMI with all other runs.*

**Your Answer:**

Code used: see **Cell 33** directly below.

(a) Pairwise stability (10 runs):
- Karate: mean NMI = **0.9214**, min = **0.6880**, max = **1.0000**.
- Football: mean NMI = **0.9629**, min = **0.9198**, max = **1.0000**.

Football is more stable (higher mean and much higher minimum NMI). Intuitively, its conference structure is stronger and less ambiguous, while karate has more boundary ambiguity in a smaller graph.

(b) Unstable nodes are those that switch community context across runs (measured by distinct community-member sets over 50 runs).

Top unstable karate nodes include:
`33, 32, 31, 8, 23, 27, 29, 30` (all with switch-rate $\approx 0.082$ in this experiment).

Top unstable football nodes include:
`LouisianaTech, MiddleTennesseeState, LouisianaLafayette, LouisianaMonroe` (switch-rate $\approx 0.061$).

Shared structural pattern: these nodes tend to sit near community boundaries (bridge-like connectors or relatively ambiguous local neighborhoods), so small stochastic differences in optimization can move them between nearby communities.

(c) One practical consensus algorithm for 100 runs:
1. Run Louvain 100 times with different seeds.
2. Compute pairwise NMI between all runs.
3. Select the **medoid partition** (the run with highest average NMI to all others).
4. Optionally break ties by higher modularity.

In this notebook run on football, the medoid example selected run index 5 with average NMI **0.9803**, modularity $Q=0.6043$, and 10 communities. This gives a representative, stable single partition without requiring arbitrary post-hoc label matching.

In [17]:
# Question 3 support code: Louvain stability details + unstable nodes + consensus exemplar

# Reuse Section 5 function for 10-run summary
q3_stab_k = community_stability(G_karate, n_runs=10)
q3_stab_fb = community_stability(G_football, n_runs=10)

print("10-run stability summary")
print(f"Karate:   mean={q3_stab_k['mean_nmi']:.4f}, min={q3_stab_k['min_nmi']:.4f}, max={q3_stab_k['max_nmi']:.4f}, unique_counts={q3_stab_k['n_unique']}")
print(f"Football: mean={q3_stab_fb['mean_nmi']:.4f}, min={q3_stab_fb['min_nmi']:.4f}, max={q3_stab_fb['max_nmi']:.4f}, unique_counts={q3_stab_fb['n_unique']}")

# Identify unstable nodes by counting how many distinct community member-sets
# each node belongs to across runs (label-invariant measure).
def node_instability_profile(G, n_runs=50):
    parts = [list(nx.community.louvain_communities(G, seed=i)) for i in range(n_runs)]
    nodes = sorted(G.nodes())

    signatures = {n: [] for n in nodes}
    for p in parts:
        for comm in p:
            fcomm = frozenset(comm)
            for n in comm:
                signatures[n].append(fcomm)

    unique_counts = {n: len(set(sigs)) for n, sigs in signatures.items()}
    switch_rates = {
        n: (unique_counts[n] - 1) / (n_runs - 1) if n_runs > 1 else 0.0
        for n in nodes
    }

    ranked = sorted(nodes, key=lambda n: (switch_rates[n], unique_counts[n], G.degree(n)), reverse=True)
    top = ranked[:8]
    return unique_counts, switch_rates, top

q3_k_uc, q3_k_sr, q3_k_top = node_instability_profile(G_karate, n_runs=50)
q3_fb_uc, q3_fb_sr, q3_fb_top = node_instability_profile(G_football, n_runs=50)

print("\nTop unstable karate nodes (node, switch_rate, degree):")
for n in q3_k_top:
    print(f"  {n:>2}  sr={q3_k_sr[n]:.3f}  deg={G_karate.degree(n)}")

print("\nTop unstable football nodes (node, switch_rate, degree):")
for n in q3_fb_top:
    print(f"  {str(n):>15}  sr={q3_fb_sr[n]:.3f}  deg={G_football.degree(n)}")

# One concrete consensus strategy: medoid partition among 100 runs.
def average_nmi_matrix(partitions):
    n = len(partitions)
    M = np.zeros((n, n), dtype=float)
    for i in range(n):
        M[i, i] = 1.0
        for j in range(i + 1, n):
            v = nmi_score(partitions[i], partitions[j])
            M[i, j] = v
            M[j, i] = v
    return M

q3_parts_100 = [list(nx.community.louvain_communities(G_football, seed=i)) for i in range(100)]
q3_M = average_nmi_matrix(q3_parts_100)
q3_avg_nmi = q3_M.mean(axis=1)
q3_medoid_idx = int(np.argmax(q3_avg_nmi))
q3_medoid = q3_parts_100[q3_medoid_idx]
q3_medoid_q = compute_modularity(G_football, q3_medoid)

print("\nConsensus-by-medoid (football, 100 runs):")
print(f"  selected run index = {q3_medoid_idx}")
print(f"  avg NMI to all runs = {q3_avg_nmi[q3_medoid_idx]:.4f}")
print(f"  modularity Q = {q3_medoid_q:.4f}")
print(f"  n_communities = {len(q3_medoid)}")

10-run stability summary
Karate:   mean=0.9214, min=0.6880, max=1.0000, unique_counts=2
Football: mean=0.9629, min=0.9198, max=1.0000, unique_counts=2

Top unstable karate nodes (node, switch_rate, degree):
  33  sr=0.082  deg=17
  32  sr=0.082  deg=12
  31  sr=0.082  deg=6
   8  sr=0.082  deg=5
  23  sr=0.082  deg=5
  27  sr=0.082  deg=4
  29  sr=0.082  deg=4
  30  sr=0.082  deg=4

Top unstable football nodes (node, switch_rate, degree):
    LouisianaTech  sr=0.061  deg=10
  MiddleTennesseeState  sr=0.061  deg=9
  LouisianaLafayette  sr=0.061  deg=8
  LouisianaMonroe  sr=0.061  deg=8
  SouthernCalifornia  sr=0.041  deg=12
          Alabama  sr=0.041  deg=11
          Arizona  sr=0.041  deg=11
     ArizonaState  sr=0.041  deg=11

Consensus-by-medoid (football, 100 runs):
  selected run index = 5
  avg NMI to all runs = 0.9803
  modularity Q = 0.6043
  n_communities = 10
